# Charity Sale Analysis Walkthrough

This notebook walks through the cleaned charity sale data used in the project.

Before running the notebook, run the full project pipeline from the repository root:

```bash
python src/run_all.py
```

The notebook uses anonymized sample data and is not an official financial report.

## 1. Setup

The setup cell locates the project root whether the notebook is opened from the repository root or from the `notebooks/` folder.

In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SUMMARY_DIR = PROJECT_ROOT / "reports" / "summary_tables"
METRICS_PATH = PROJECT_ROOT / "models" / "model_metrics.json"

required_files = [
    PROCESSED_DIR / "cleaned_donations.csv",
    PROCESSED_DIR / "cleaned_inventory.csv",
    PROCESSED_DIR / "cleaned_sales.csv",
    PROCESSED_DIR / "cleaned_booth_layout.csv",
    SUMMARY_DIR / "booth_summary.csv",
    SUMMARY_DIR / "estimate_vs_actual_summary.csv",
]

missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    print("Processed files are missing. Run `python src/run_all.py` first.")
    for path in missing_files:
        print(f"Missing: {path.relative_to(PROJECT_ROOT)}")
else:
    print("All required processed files are available.")

## 2. Load Cleaned Data

The project keeps raw sample records and processed cleaned records separate. The notebook reads from `data/processed/`.

In [ ]:
donations = pd.read_csv(PROCESSED_DIR / "cleaned_donations.csv")
inventory = pd.read_csv(PROCESSED_DIR / "cleaned_inventory.csv")
sales = pd.read_csv(PROCESSED_DIR / "cleaned_sales.csv")
booth_layout = pd.read_csv(PROCESSED_DIR / "cleaned_booth_layout.csv")

print(f"Donation records: {len(donations)}")
print(f"Inventory records: {len(inventory)}")
print(f"Sale records: {len(sales)}")
print(f"Booth records: {len(booth_layout)}")

## 3. Donation Analysis

This section summarizes direct donations by donor type and team.

In [ ]:
donation_by_type = donations.groupby("donor_type", as_index=False).agg(
    donation_amount_cny=("donation_amount_cny", "sum"),
    donation_count=("donation_id", "count"),
)
donation_by_team = donations.groupby("team", as_index=False).agg(
    donation_amount_cny=("donation_amount_cny", "sum"),
    donation_count=("donation_id", "count"),
)

total_direct_donations = donations["donation_amount_cny"].sum()
donation_by_type

## 4. Inventory Analysis

Inventory records show item categories, estimated values, condition, teams, booth areas, and post-event status.

In [ ]:
inventory_summary = inventory.groupby("item_category", as_index=False).agg(
    total_quantity=("quantity", "sum"),
    estimated_total_value_cny=("estimated_total_value_cny", "sum"),
)
inventory_status = inventory.groupby("status", as_index=False).agg(
    item_groups=("item_id", "count"),
    total_quantity=("quantity", "sum"),
)

inventory_summary.sort_values("total_quantity", ascending=False).head(10)

## 5. Sales Analysis

Sale records are connected to inventory through item IDs.

In [ ]:
sales_summary = sales.groupby("item_category", as_index=False).agg(
    quantity_sold=("quantity_sold", "sum"),
    sale_revenue_cny=("total_sale_cny", "sum"),
)

total_sale_revenue = sales["total_sale_cny"].sum()
total_funds = total_direct_donations + total_sale_revenue

print(f"Total direct donations: ¥{total_direct_donations:,.0f}")
print(f"Total sale revenue: ¥{total_sale_revenue:,.0f}")
print(f"Total funds raised: ¥{total_funds:,.0f}")
sales_summary.sort_values("sale_revenue_cny", ascending=False).head(10)

## 6. Booth Layout Analysis

Booth-level review connects planning fields with the generated revenue summary.

In [ ]:
booth_summary = pd.read_csv(SUMMARY_DIR / "booth_summary.csv")
booth_summary[
    [
        "booth_area",
        "assigned_team",
        "main_category",
        "estimated_items",
        "actual_items",
        "booth_revenue_cny",
    ]
]

## 7. Estimate vs Actual Price

This table compares estimated sold value with actual sale revenue by category.

In [ ]:
estimate_vs_actual = pd.read_csv(SUMMARY_DIR / "estimate_vs_actual_summary.csv")
estimate_vs_actual[
    [
        "item_category",
        "estimated_sold_value_cny",
        "actual_sale_total_cny",
        "price_difference_cny",
    ]
]

## 8. Simple Machine Learning Results

The model metrics are useful for learning, but the dataset is too small for official decision-making.

In [ ]:
if METRICS_PATH.exists():
    metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
else:
    metrics = {}

metrics

## 9. Key Findings

- Direct donations made up the largest share of total funds.
- Item IDs made it possible to connect inventory and sale records.
- Booth-level summaries helped review item allocation and display planning.
- Estimate vs actual comparison helped reflect on pricing choices.
- The machine learning results are exploratory learning outputs.

## 10. Reflection

Before this project, I thought charity sale work was mostly about collecting and selling items. After working on the data side, I learned that clear records, item categories, estimated values, and simple analysis can make an event easier to organize and improve.

This project helped connect community service, operations, and data analysis in a practical way.